# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Charanya207/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
My rule: Use CTR and content freshness to calculate an action score. Higher CTR and fresher content should receive a higher score.

Reason codes:
HIGH_CTR — strong user engagement.
FRESH_CONTENT — recently updated content.
LOW_CTR — weak user engagement.
STALE_CONTENT — older content.# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import pandas as pd
import os

# Load the internship dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Convert required columns
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce").fillna(0)

# Find the date column
date_col = next(
    (c for c in df.columns if c.lower() in ["date", "published_at", "content_date", "updated_at"]),
    None
)

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    max_date = df[date_col].max()
    age_days = (max_date - df[date_col]).dt.days.fillna(0).clip(lower=0)
    freshness = 1 / (1 + age_days)
else:
    freshness = pd.Series(1.0, index=df.index)

# Normalize CTR between 0 and 1
ctr_min = df["ctr"].min()
ctr_max = df["ctr"].max()

if ctr_max > ctr_min:
    ctr_score = (df["ctr"] - ctr_min) / (ctr_max - ctr_min)
else:
    ctr_score = pd.Series(0.0, index=df.index)

# Action score: higher CTR + fresher content = higher score
df["action_score"] = 0.7 * ctr_score + 0.3 * freshness

# Rank everything
df = df.sort_values("action_score", ascending=False).reset_index(drop=True)
df["rank"] = df.index + 1

# Create output folder and save CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(df))
print("Output:", output_path)

display(df.head(20))


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.